# Turning Text into Research Data

### Measurement and inference with LLMs

**SAFE Data Seminar · hands-on notebook**

You have a corpus of 4,000 firm-quarter disclosures. You want to know whether
filings that discuss funding stress are followed by wider CDS spreads. Reading
4,000 filings is not happening, so you have an LLM read them for you.

This notebook is about what that does to your coefficient, and what to do about it.

---

### How to use this in the session

We go through this **together** — I run every cell on screen, so nothing here
depends on your own setup working. If your installation is broken, just watch; the
notebook is yours to keep and runs in about ten seconds afterwards.

| | |
|---|---|
| **Sections 1–3** | We run together. See the problem appear. |
| **Exercise 1** | You try it (2 minutes), then we do it together. |
| **Exercise 2** | Guess first, then we run it. |
| **Exercise 3** | Take-home — the one that applies to your own project. |

**Requirements:** `numpy`, `pandas`, `matplotlib`. No API key, no internet, no
paid model access.

---

### A word on the data

**The corpus in this notebook is simulated, not real.** That is deliberate, and it
is the only way the exercise works: because we generate the data, we know the true
coefficient is exactly **β = 2.0**, so we can see precisely how far each estimator
lands from it. With real filings you never get to check your answer.

Section 8 shows how to swap in your own corpus once you have seen the mechanics.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(20261102)
pd.set_option("display.float_format", lambda v: f"{v:8.3f}")

BETA_TRUE  = 2.0    # <- the answer. In real work you never see this.
GAMMA_TRUE = 1.0
ALPHA_TRUE = 0.5
SIGMA      = 3.0
P_D        = 0.30   # share of filings that genuinely discuss funding stress

## 2. The data-generating process

A latent binary concept $D^*_i$ — *"this filing genuinely discusses funding stress"* —
moves the outcome $y_i$ (change in CDS spread, bps):

$$y_i = \alpha + \beta D^*_i + \gamma x_i + \varepsilon_i$$

We observe $y_i$ and a covariate $x_i$ for everything. We **do not** observe $D^*_i$
at scale. We observe an LLM's guess at it.

The two prompts below are the kind of pair you would actually write — neither is
unreasonable, and you would have no way to tell them apart without ground truth:

| | sensitivity | specificity | character |
|---|---|---|---|
| **Prompt A** "Does this filing mention any funding or liquidity pressure?" | 0.95 | 0.80 | catches nearly everything, over-flags |
| **Prompt B** "Does this filing state that the firm faces funding difficulty?" | 0.65 | 0.98 | almost never over-flags, misses a third |

In [ ]:
PROMPTS = {"A_broad": (0.95, 0.80), "B_strict": (0.65, 0.98)}

def draw(rng, n, differential=False):
    """One draw of the corpus. Returns truth, LLM labels, outcome, covariate."""
    x      = rng.normal(size=n)
    d_true = (rng.random(n) < P_D).astype(int)
    y      = ALPHA_TRUE + BETA_TRUE * d_true + GAMMA_TRUE * x + rng.normal(0, SIGMA, n)

    out = {"x": x, "d_true": d_true, "y": y}
    for name, (sens, spec) in PROMPTS.items():
        if differential:
            # errors depend on the OUTCOME, not just on the concept:
            # the model over-flags exactly the filings that ended badly
            shift = 0.12 * np.sign(y - np.median(y))
            s_eff = np.clip(sens + shift, 0.01, 0.99)
            p_eff = np.clip(spec - shift, 0.01, 0.99)
        else:
            s_eff, p_eff = np.full(n, sens), np.full(n, spec)
        u = rng.random(n)
        out[name] = np.where(d_true == 1, u < s_eff, u > p_eff).astype(int)
    return pd.DataFrame(out)

corpus  = draw(rng, 4000)   # the LLM reads these
labeled = draw(rng, 200)    # you read these yourself

corpus.head()

### How good are the labels?

Before looking at any regression — just how well do the two prompts agree with
ground truth? This is the diagnostic most papers report, and in a moment we will
see why it is not nearly enough.

In [ ]:
def label_quality(df):
    rows = []
    for p in PROMPTS:
        sens = df.loc[df.d_true == 1, p].mean()
        spec = 1 - df.loc[df.d_true == 0, p].mean()
        rows.append({"prompt": p,
                     "share flagged": df[p].mean(),
                     "sensitivity": sens,
                     "specificity": spec,
                     "raw agreement": (df[p] == df.d_true).mean(),
                     "Cohen kappa": cohen_kappa(df[p], df.d_true)})
    return pd.DataFrame(rows)

def cohen_kappa(a, b):
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe)

print(f"true share discussing funding stress: {corpus.d_true.mean():.3f}\n")
label_quality(corpus)

> **Pause here.** Both prompts look defensible: 85% and 88% raw agreement, kappa in
> the range people routinely publish. If you saw these numbers in a referee report
> you would probably wave them through.
>
> **Write down a guess before running the next cell:** you regress `y` on each LLM
> label. How far from 2.0 do you expect to land — and will the two prompts give you
> roughly the same answer?

## 3. The naive regression

> **The helper below is a black box — you do not need to read inside it.**
>
> `ols_if` regresses `y` on `d` and `x` and returns two things: the coefficient
> on `d`, and an ingredient we need later to compute a standard error (its
> *influence function*). Nothing you do in this notebook requires understanding
> the second one. It is simply what lets us combine two different samples
> correctly in section 5.
>
> *If you want the detail:* the influence function is each observation's
> contribution to the estimator, and its variance divided by `n` gives the
> heteroskedasticity-robust variance of the coefficient. If that sentence did
> nothing for you, skip it — the rest of the notebook is unaffected.

In [ ]:
def ols_if(y, d, x):
    """
    OLS of y on [1, d, x]. Returns the coefficient on d and its influence
    function. Treat as a black box -- see the note above.
    """
    n   = len(y)
    Z   = np.column_stack([np.ones(n), d, x]).astype(float)
    ZtZ = Z.T @ Z
    beta  = np.linalg.solve(ZtZ, Z.T @ y)
    resid = y - Z @ beta
    Sinv  = np.linalg.inv(ZtZ / n)
    psi   = (Z * resid[:, None]) @ Sinv[:, 1]
    return float(beta[1]), psi

def report(beta, se, label, z=1.96):
    lo, hi = beta - z * se, beta + z * se
    flag = "contains 2.0" if lo <= BETA_TRUE <= hi else "MISSES 2.0"
    print(f"{label:34s} beta={beta:6.3f}  se={se:5.3f}  "
          f"95% CI [{lo:6.3f}, {hi:6.3f}]   {flag}")

print(f"{'TRUTH':34s} beta={BETA_TRUE:6.3f}\n")

for p in PROMPTS:
    b, psi = ols_if(corpus.y.values, corpus[p].values, corpus.x.values)
    report(b, psi.std(ddof=1) / np.sqrt(len(corpus)), f"naive, prompt {p} (n=4,000)")

Two things to notice, and the second is the one that should bother you:

1. **Both confidence intervals exclude the truth.** Not marginally — comfortably.
   With 4,000 observations the naive CI is tight, which makes the problem worse,
   not better: you get a precisely estimated wrong number.
2. **The two prompts disagree with each other** by more than either CI is wide.
   Nothing changed except the wording of the question you asked the model.

This is the central result in Ludwig, Mullainathan and Rambachan (2026): without a
validation sample, seemingly innocuous choices — which model, which prompt — move
the parameter estimate around, and you have no way to detect it.

## 4. Why — and the fix

Measurement error in a *binary regressor* is not classical. With misclassification
rates $\alpha_0 = \Pr(\hat D = 1 \mid D^* = 0)$ and $\alpha_1 = \Pr(\hat D = 0 \mid D^* = 1)$,
the naive OLS coefficient converges to something like

$$\hat\beta \;\xrightarrow{p}\; \beta \cdot \underbrace{(1 - \alpha_0 - \alpha_1)}_{\text{attenuation}} \cdot \frac{\operatorname{Var}(D^*)}{\operatorname{Var}(\hat D)}$$

which is why both estimates came in *below* 2.0. But do not reach for that formula.
It holds only when the LLM's errors are unrelated to the outcome — and in section 6
we will break exactly that assumption and watch the bias flip sign.

**The general fix needs no assumption about the error structure at all.** Hand-label
a small random sample. On it you can see both the truth and the LLM's guess, so you
can measure the bias directly:

$$\widehat{\text{bias}} \;=\; \hat\beta^{\text{labeled}}(\text{LLM label}) - \hat\beta^{\text{labeled}}(\text{truth})$$

and subtract it from the full-corpus estimate:

$$\boxed{\;\hat\beta^{\text{rect}} \;=\; \underbrace{\hat\beta^{\text{corpus}}(\text{LLM label})}_{\text{precise, biased}} \;-\; \widehat{\text{bias}}\;}$$

This is the **rectifier** from prediction-powered inference (Angelopoulos et al., 2023),
and it is the same idea as design-based supervised learning in political science. The
only thing it needs is that the hand-labeled set be a *random* sample of the corpus.

Because the two samples are independent, the variances simply add:

$$\operatorname{Var}(\hat\beta^{\text{rect}}) = \operatorname{Var}\!\big(\hat\beta^{\text{corpus}}\big) + \operatorname{Var}\!\big(\widehat{\text{bias}}\big)$$

---

## Exercise 1 — implement the rectifier

⏱️ **Two minutes on your own, then we do it together.**

Fill in the three lines marked `TODO`, using `ols_if` — it hands you back both
things you need. The solution is in the cell after this one, so do not scroll if
you want to try it first.

*Hint for (c):* the point estimate is the corpus estimate **minus** the bias you
measured. For the standard error, the two samples are independent, so the
**variances add** — add them, then take the square root.

In [ ]:
def rectified(corpus, labeled, prompt, z=1.96):
    # full corpus, LLM label: precise but biased
    b_corpus, psi_corpus = ols_if(corpus.y.values, corpus[prompt].values, corpus.x.values)
    se_corpus = psi_corpus.std(ddof=1) / np.sqrt(len(corpus))

    # TODO (a): same regression on the LABELED set, using the LLM label
    b_lab_llm, psi_lab_llm = ...

    # TODO (b): same regression on the LABELED set, using the TRUTH (d_true)
    b_lab_true, psi_lab_true = ...

    # TODO (c): combine. Point estimate = corpus estimate minus the measured bias.
    #           Variance = corpus variance + variance of the bias term.
    beta = ...
    se   = ...

    return beta, se

# rectified(corpus, labeled, "A_broad")

### Solution — we write this one together

In [ ]:
def rectified(corpus, labeled, prompt, z=1.96):
    b_corpus, psi_corpus = ols_if(corpus.y.values, corpus[prompt].values, corpus.x.values)
    se_corpus = psi_corpus.std(ddof=1) / np.sqrt(len(corpus))

    b_lab_llm,  psi_lab_llm  = ols_if(labeled.y.values, labeled[prompt].values,  labeled.x.values)
    b_lab_true, psi_lab_true = ols_if(labeled.y.values, labeled.d_true.values,   labeled.x.values)

    bias    = b_lab_llm - b_lab_true
    se_bias = (psi_lab_llm - psi_lab_true).std(ddof=1) / np.sqrt(len(labeled))

    beta = b_corpus - bias
    se   = np.sqrt(se_corpus**2 + se_bias**2)
    return beta, se


print(f"{'TRUTH':34s} beta={BETA_TRUE:6.3f}\n")

b_lab, psi_lab = ols_if(labeled.y.values, labeled.d_true.values, labeled.x.values)
report(b_lab, psi_lab.std(ddof=1)/np.sqrt(len(labeled)), "hand-labeled only (n=200)")

for p in PROMPTS:
    b, psi = ols_if(corpus.y.values, corpus[p].values, corpus.x.values)
    report(b, psi.std(ddof=1)/np.sqrt(len(corpus)), f"naive, prompt {p}")
for p in PROMPTS:
    report(*rectified(corpus, labeled, p), f"rectified, prompt {p}")

Three things changed:

- Both rectified intervals now contain the truth.
- The two prompts now **agree with each other**. The choice that mattered enormously
  in section 3 has stopped mattering. That is the real payoff — not a nicer point
  estimate, but a result that no longer depends on a wording decision you made
  without thinking.
- The rectified interval is narrower than hand-labeling alone, but only somewhat.
  Hold that thought; section 7 is about when that gain is real and when it is not.

## 5. Does the confidence interval mean what it says?

One draw proves nothing — we could have got lucky. The honest check is to repeat the
whole exercise many times and ask how often each 95% interval actually contains 2.0.

(≈30 seconds.)

In [ ]:
def monte_carlo(reps=500, differential=False, seed=7):
    rng = np.random.default_rng(seed)
    acc = {}
    for _ in range(reps):
        c = draw(rng, 4000, differential)
        l = draw(rng, 200,  differential)
        cand = {}
        b, psi = ols_if(l.y.values, l.d_true.values, l.x.values)
        cand["hand-labeled only"] = (b, psi.std(ddof=1)/np.sqrt(len(l)))
        for p in PROMPTS:
            b, psi = ols_if(c.y.values, c[p].values, c.x.values)
            cand[f"naive {p}"]     = (b, psi.std(ddof=1)/np.sqrt(len(c)))
            cand[f"rectified {p}"] = rectified(c, l, p)
        for k, (b, se) in cand.items():
            a = acc.setdefault(k, {"b": [], "cov": 0})
            a["b"].append(b)
            a["cov"] += int(abs(b - BETA_TRUE) <= 1.96 * se)
    rows = []
    for k, a in acc.items():
        b = np.array(a["b"])
        rows.append({"estimator": k, "mean beta": b.mean(),
                     "bias": b.mean() - BETA_TRUE,
                     "RMSE": np.sqrt(np.mean((b - BETA_TRUE)**2)),
                     "CI coverage": a["cov"] / reps})
    return pd.DataFrame(rows).set_index("estimator")

monte_carlo()

A 95% interval that contains the truth 0% of the time is not a confidence interval.
It is a number with decorations.

## 6. When the formula you reached for is wrong

Section 4 gave an attenuation formula and then told you not to use it. Here is why.

Suppose the LLM over-flags *exactly the filings that ended badly* — plausible whenever
the text and the outcome share a cause, which in finance is most of the time. The
errors are now **differential**: they depend on `y`, not just on the concept.

In [ ]:
pd.concat({"errors unrelated to outcome": monte_carlo(reps=500),
           "errors correlated with outcome": monte_carlo(reps=500, differential=True, seed=99)},
          names=["regime"])

The bias **flipped sign** — it is now upward, not attenuation. Any correction derived
from "measurement error attenuates coefficients" would have pushed your estimate
further from the truth, confidently.

The rectifier does not care. It never assumed anything about the error structure; it
just measured the bias where it could see it. That is the whole argument for spending
a day hand-labeling 200 documents.

## 7. How many documents, and is it even worth it?

---

## Exercise 2 — find the break-even point

⏱️ **Guess first, in the chat. Then we run it.**

The rectified estimator combines a big noisy sample with a small clean one.
Intuition says that must beat using the small clean one alone. **Intuition is
wrong**, and the next cell shows where it breaks.

**Your guess:** below what label accuracy does using the model stop being worth
it at all?

In [ ]:
def quality_sweep(accuracies=(0.60, 0.70, 0.80, 0.90, 0.95, 0.99), reps=200, seed=13):
    global PROMPTS
    keep, rng, rows = PROMPTS, np.random.default_rng(seed), []
    for acc in accuracies:
        PROMPTS = {"Q": (acc, acc)}
        wr, wl = [], []
        for _ in range(reps):
            c, l = draw(rng, 4000), draw(rng, 200)
            _, se_r = rectified(c, l, "Q")
            _, psi  = ols_if(l.y.values, l.d_true.values, l.x.values)
            wr.append(2 * 1.96 * se_r)
            wl.append(2 * 1.96 * psi.std(ddof=1) / np.sqrt(len(l)))
        rows.append({"label accuracy": acc,
                     "CI width, rectified": np.mean(wr),
                     "CI width, hand-labeled only": np.mean(wl),
                     "% narrower": 100 * (1 - np.mean(wr) / np.mean(wl))})
    PROMPTS = keep
    return pd.DataFrame(rows)

sweep = quality_sweep()
sweep

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = sweep["label accuracy"] * 100
ax.plot(x, sweep["CI width, hand-labeled only"], "-o", color="#898781", lw=2, ms=7,
        label="hand-labeled only (n=200)")
ax.plot(x, sweep["CI width, rectified"], "-o", color="#2a78d6", lw=2, ms=7,
        label="rectified (LLM + 200 hand labels)")
ax.set_xlabel("LLM label accuracy (%)")
ax.set_ylabel("width of the 95% CI")
ax.set_title("A bad label is worse than no label", fontweight="bold")
ax.legend(frameon=False)
ax.grid(axis="y", color="#e1e0d9")
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

Below roughly **78% accuracy the rectified estimator is *wider* than ignoring the LLM
entirely**. A noisy proxy contributes a noisy bias estimate, and that noise has to be
carried in the standard error.

> ⚠️ **78% is not a universal constant.** It is a property of *this* setup — 4,000
> documents, 200 hand labels, this effect size and base rate. Change any of them
> and the crossover moves. What generalises is the *shape*: there is always a
> crossover, and below it a poor label costs you precision rather than buying it.
> Find yours with the code above before relying on a number.

So the decision rule is not "use an LLM" or "don't". It is:

- **Label quality below ~80%** → the LLM is not buying you anything. Either improve the
  prompt until it does, or hand-label a larger sample and skip the model.
- **Label quality above ~90%** → substantial gains, and they grow steeply.
- **Either way you need the validation sample**, because without it you cannot tell
  which regime you are in. That is the part that is not optional.

---

## Exercise 3 — take-home

🏠 **The one that applies to your own project.** We will not do this live.

Modify `quality_sweep` to vary the *size* of the hand-labeled set (25, 50, 100,
200, 400, 800) at fixed 90% label accuracy, and plot CI width against it. Where do
the returns to another day of labeling flatten out?

Then do it again with the parameters of a project of your own: your corpus size,
your plausible effect size, your base rate. The answer tells you how many
documents to budget for.

In [ ]:
# your turn
#
# starting point: copy quality_sweep, replace the accuracies loop with a loop
# over n_lab values, and pass n_lab into the second draw() call:
#
#     c, l = draw(rng, 4000), draw(rng, n_lab)
#
# keep PROMPTS fixed at {"Q": (0.90, 0.90)} throughout.

## 8. Bringing your own corpus — the protocol

The mechanics above do not change. What changes is how you get `d_true` and the LLM
label. A checklist:

**Draw the validation sample first, and draw it at random.** Not the first 200, not
the interesting ones, not the ones where the model was unsure. The entire validity of
the rectifier rests on this one step. Draw it before you look at any model output, so
you cannot be tempted.

**Label it yourself, blind to the model's answer.** If you label while looking at the
LLM's guess, you are measuring your own agreement with the model, not the truth. Have
a second person label a subset and report the disagreement rate honestly.

**Write the codebook before labeling, and ship it as the prompt.** If you cannot write
down a rule precise enough for an RA to follow, the concept is not well enough defined
for a model either — and the prompt sensitivity you saw in section 3 is a symptom of
exactly that.

**Cache every model call to disk, keyed by document id, prompt hash, and model version.**
Model versions get deprecated and results stop replicating. The cache *is* your
replication package.

```python
import hashlib, json, pathlib

CACHE = pathlib.Path("llm_cache.jsonl")

def label_document(doc_id, text, prompt, model="<model-name>"):
    key = hashlib.sha256(f"{doc_id}|{prompt}|{model}".encode()).hexdigest()
    for line in CACHE.read_text().splitlines() if CACHE.exists() else []:
        rec = json.loads(line)
        if rec["key"] == key:
            return rec["label"]
    label = call_your_api(text, prompt, model)      # <- your provider here
    with CACHE.open("a") as f:
        f.write(json.dumps({"key": key, "doc_id": doc_id, "prompt": prompt,
                            "model": model, "label": label}) + "\n")
    return label
```

**Run at least two prompts and report both.** If the rectified estimates diverge, your
concept is underspecified and no amount of correction will save it. This is cheap
insurance and almost nobody does it.

**On leakage, if you are predicting rather than measuring.** Everything above concerns
*estimation* — automating a measurement for downstream analysis. The other use is
*prediction*: forecasting an outcome from text. There the binding constraint is
different, and it is that the model may have seen your outcome during training. A
model asked to predict 2021 returns from a 2021 filing has, in some sense, already
read the answer. Guarding against this means choosing a model with a training cutoff
before your sample period, or designing around it — not validating harder.

## Where to read more

- **Ludwig, Mullainathan & Rambachan (2026)**, "Large Language Models: An Applied
  Econometric Framework," *Annual Review of Economics* 18, 283–316. The framework
  this notebook implements; the prediction/estimation split and the validation-sample
  argument are theirs.
- **Dell & Rambachan (2026)**, NBER Summer Institute Methods Lectures, "Estimation and
  Inference with AI-Generated Data." Video and slides free on the NBER site. The fuller
  treatment, including the cases this notebook skips.
- **Angelopoulos et al. (2023)**, "Prediction-Powered Inference," *Science* 382. The
  rectifier in its general form, beyond OLS.
- **Egami et al.**, design-based supervised learning — the same correction arrived at
  independently in political science, with useful discussion of the labeling protocol.